## **Build a QSAR model for Dopamine Receptor D2 inhibitors**


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !uv pip install --system rdkit pandas datamol molfeat numpy scikit-learn yellowbrick wget

In [ ]:
import pandas as pd
import datamol as dm
from molfeat.calc import FPCalculator
from molfeat.trans import MoleculeTransformer
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from yellowbrick.regressor import prediction_error, residuals_plot
import wget

In [ ]:
wget.download("https://raw.githubusercontent.com/Babakmamnoon/QSAR_Modeling_for_Dopamine_Receptor_D2_Inhibitors/refs/heads/main/DRD2.csv")

## Read the data into a [Pandas](https://pandas.pydata.org/) dataframe

In [ ]:
filename = "DRD2.csv"
df = pd.read_csv(filename)
activity_col = df.columns[-1]
print(f"The activity column is probably {activity_col}")
df.head()

## Instantiate a Fingerprint calculator from the [molfeat](https://m2d2.io/blog/posts/introducing-molfeat-a-hub-of-molecular-featurizers/) package

In [ ]:
calc = FPCalculator("ecfp")

## Instantiate a molecule transfomer from molfeat.    
This object takes a list of SMILES as input and returns descriptors.  It's very flexible and can run in parallel

In [ ]:
trans = MoleculeTransformer(calc)

## Calculate the fingerprints.    
Note the use of the function from [datamol](https://datamol.io) that silences logging messages from the RDKit.  This is more polite version of my rd_shut_the_hell_up function in [useful_rdkit_utils](https://github.com/PatWalters/useful_rdkit_utils).

In [ ]:
%%time
with dm.without_rdkit_log():
    df['fp'] = trans.transform(df.SMILES.values)

## Split the data into training and test sets.  
I like to do this with dataframes.  That way I don't have to remember the order in which `train_X, test_X, train_y, and test_y` are returned by [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

In [ ]:
train, test = train_test_split(df)

## Instantiate an sklearn style regressor.  
In this case I used [HistGradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html), which is the scikit-learn implementation of [LightGBM](https://lightgbm.readthedocs.io/en/latest/Python-Intro.html)

In [ ]:
model = HistGradientBoostingRegressor()

## **8.** Use [YellowBrick](https://www.scikit-yb.org/en/latest/) to build a model and visualize its performance.
The **Loss** reported in the plot below is the [$R^2$](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html) for the model.

In [ ]:
%%time
visualizer = prediction_error(model,np.stack(train.fp),train[activity_col],np.stack(test.fp),test[activity_col])

## Residual plot
Plot the residuals for the training and test sets

In [ ]:
viz = residuals_plot(model,np.stack(train.fp), train[activity_col], np.stack(test.fp), test[activity_col], is_fitted=True)